# Closed-Loop Active Evidential Sensing & Rescue Routing Simulation
This notebook runs closed-loop tracking simulations of targets drifted by ocean currents using a dynamic evidential Kalman Filter and schedules resource allocation using a Risk-UCB router.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add parent folder to path to allow importing from src
sys.path.append(os.path.abspath(os.path.join('..')))

from src.simulator import EvidentialClassifierSimulator
from src.tracker import UncertaintyKalmanFilter
from src.simulation import run_simulation

In [ ]:
sim_classifier = EvidentialClassifierSimulator()
uav_pos = np.array([0.0, 0.0])
uav_speed = 6.0
ocean_current = np.array([0.1, -0.05])
victims = {
    1: {"state": np.array([60.0, 45.0, 0.0, 0.0]), "class": 0, "occluded": True, "decay": 0.08, "name": "Victim A (Drowning, Occluded)"},
    2: {"state": np.array([30.0, -20.0, 0.0, 0.0]), "class": 2, "occluded": False, "decay": 0.02, "name": "Victim B (Swimming, Clear)"}
}
trackers = {i: UncertaintyKalmanFilter() for i in victims}
states = {i: victims[i]["state"].copy() for i in victims}

print("Starting Step-by-Step Simulation Routing...")
for step in range(1, 6):
    print(f"\n--- Step {step} | UAV: ({uav_pos[0]:.1f}, {uav_pos[1]:.1f}) ---")
    priorities = {}
    target_positions = {}
    for vid, data in victims.items():
        data["state"][:2] += ocean_current + np.random.normal(0, 0.1, 2)
        true_pos = data["state"][:2]
        dist = np.linalg.norm(true_pos - uav_pos)
        
        spatial_cov = np.eye(2) * ((0.5 + 0.01 * dist) ** 2)
        meas = true_pos + np.random.multivariate_normal([0, 0], spatial_cov)
        
        beliefs, u, probs = sim_classifier.estimate(data["class"], dist, data["occluded"])
        states[vid] = trackers[vid].update(trackers[vid].predict(states[vid], ocean_current), meas, spatial_cov, u)
        
        exp_risk = np.sum(probs * sim_classifier.risk_weights)
        p_raw = exp_risk + 0.8 * u * (1.0 - exp_risk)
        travel_time = dist / uav_speed
        priority_score = (p_raw * np.exp(data["decay"] * travel_time)) / (dist + 1.0)
        
        priorities[vid] = priority_score
        target_positions[vid] = states[vid][:2]
        print(f"  Target {vid} ({data['name'][:10]}): Dist={dist:.1f}m | Uncertainty={u:.3f} | Priority={priority_score:.3f}")
        
    best_target = max(priorities, key=priorities.get)
    print(f"  ==> Active Allocation: Flying toward Target {best_target} ({victims[best_target]['name']})")
    direction = target_positions[best_target] - uav_pos
    d_norm = np.linalg.norm(direction)
    if d_norm <= uav_speed:
        uav_pos = target_positions[best_target].copy()
        victims[best_target]["occluded"] = False
    else:
        uav_pos += (direction / d_norm) * uav_speed

## Comparative Analysis: Risk-UCB vs Baselines
We compare our full `aes_rarr` implementation against expected-risk `deterministic` and a `distance_router` first-nearest routing solver.

In [ ]:
print("Running 30-step comparative simulation under seed=42...")
np.random.seed(42); rows_aes,  ba    = run_simulation("aes_rarr")
np.random.seed(42); rows_det,  bd    = run_simulation("deterministic")
np.random.seed(42); rows_dist, bdist = run_simulation("distance_router")

df = pd.DataFrame(rows_aes + rows_det + rows_dist)
print(df.to_string(index=False))

# Aggregate survival summaries
agg = (df.groupby("Mode")["VSR"].mean()
         .reset_index()
         .rename(columns={"VSR": "Mean VSR"})
         .sort_values("Mean VSR", ascending=False))
rescued = (df.groupby("Mode")["Rescued"]
             .apply(lambda s: (s == "Yes").sum())
             .reset_index()
             .rename(columns={"Rescued": "Victims Rescued"}))
summary = agg.merge(rescued, on="Mode")
print("\n=== Comparative Summary ===")
print(summary.to_string(index=False))